# Project B — Teachable Machine
## Task B1: Image Model — Waste Sorting Assistant

**Objective:** Load the exported Teachable Machine image model in Python, classify webcam frames, and build a useful application (waste type → correct bin + CSV logger).

**Model:** 10 classes — trash, plastic, clothes, shoes, paper, cardboard, biological, battery, glass, metal
**Input:** 224×224×3 image, normalised to [-1, 1]  |  **Output:** 10 softmax probabilities

### Step 1 — Imports
Teachable Machine `.h5` files are Keras-2 files, so we use `tf_keras` (works on every TensorFlow version).

`pip install tensorflow tf_keras opencv-python numpy matplotlib pandas`

In [4]:
import os, csv, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tf_keras as keras

os.makedirs("output", exist_ok=True)

### Step 2 — Load model and labels

In [7]:
model = keras.models.load_model("models/image/keras_model.h5", compile=False)
labels = [line.strip().split(" ", 1)[1] for line in open("models/image/labels.txt")]
print("Classes:", labels)

Classes: ['trash', 'plastic', 'clothes', 'shoes', 'paper', 'cardboard', 'biological', 'battery', 'glass', 'metal']


### Step 3 — Model introspection (Task B4.4)
How many layers? How many trainable parameters?

In [8]:
model.summary()
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Layers      :", len(model.layers))
print("Total params:", model.count_params())
print("Trainable   :", sum(int(np.prod(w.shape)) for w in model.trainable_weights))

Model: "sequential_38"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_35 (Sequential)  (None, 1280)              410208    
                                                                 
 sequential_37 (Sequential)  (None, 10)                129100    
                                                                 
Total params: 539308 (2.06 MB)
Trainable params: 525228 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________
Input shape : (None, 224, 224, 3)
Output shape: (None, 10)
Layers      : 2
Total params: 539308
Trainable   : 525228


**Observation:** The model has just 2 top-level layers — a pre-trained MobileNet feature extractor (truncated, 410,208 params) and a small Dense head (129,100 params). Only the Dense head is trained on our custom classes; the MobileNet part was pre-trained on ImageNet. This is **Transfer Learning via Feature Extraction**.

### Step 4 — Preprocessing + prediction function
Resize to 224×224 and normalise to [-1, 1] — exactly what Teachable Machine did in training.

In [9]:
def preprocess(frame_bgr):
    img = cv2.resize(frame_bgr, (224, 224), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.asarray(img, dtype=np.float32).reshape(1, 224, 224, 3)
    return (img / 127.5) - 1.0

def predict(frame_bgr):
    pred = model.predict(preprocess(frame_bgr), verbose=0)
    idx = int(np.argmax(pred))
    return labels[idx], float(pred[0][idx]), pred[0]

### Step 5 — Test on one image
Put any photo in the `output/` folder (e.g. `output/test.jpg`) and change the path below.

In [10]:
path = "output/test.jpg"
if os.path.exists(path):
    img = cv2.imread(path)
    name, conf, probs = predict(img)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name} ({conf*100:.1f}%)")
    plt.axis("off")
    plt.show()
    plt.figure(figsize=(8, 3))
    plt.bar(labels, probs)
    plt.xticks(rotation=45)
    plt.title("Class probabilities")
    plt.show()
else:
    print("Add a test image at", path)

Add a test image at output/test.jpg


### Step 6 — Useful application: Waste Sorting Assistant (live webcam)
Shows the detected waste type, a confidence bar, and **which bin** it goes in. Every 5 seconds the detection is logged to `output/waste_log.csv`.

**How to use:** run the cell, hold a waste item in front of the camera.
**Keys:** `s` = save screenshot to `output/b1_image_result.jpg`, `q` = quit.

| Bin | Items |
|---|---|
| Recyclable | plastic, paper, cardboard, glass, metal |
| Organic | biological |
| Hazardous | battery |
| General | trash, clothes, shoes |

In [11]:
bins = {
    "plastic": "Recyclable", "paper": "Recyclable", "cardboard": "Recyclable",
    "glass": "Recyclable", "metal": "Recyclable",
    "biological": "Organic", "battery": "Hazardous",
    "trash": "General", "clothes": "General", "shoes": "General",
}

log_file = open("output/waste_log.csv", "a", newline="")
log = csv.writer(log_file)
last_log = 0

cap = cv2.VideoCapture(0)   # change to 1 if the wrong camera opens
while True:
    ok, frame = cap.read()
    if not ok:
        break

    name, conf, _ = predict(frame)
    bin_name = bins.get(name, "General")

    cv2.putText(frame, f"{name} ({conf*100:.0f}%) -> {bin_name} bin", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.rectangle(frame, (10, 45), (10 + int(300 * conf), 65), (0, 255, 0), -1)

    if time.time() - last_log > 5:
        log.writerow([time.strftime("%H:%M:%S"), name, round(conf, 2), bin_name])
        log_file.flush()
        last_log = time.time()

    cv2.imshow("Waste Sorting Assistant", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("s"):
        cv2.imwrite("output/b1_image_result.jpg", frame)
    if key == ord("q"):
        break

cap.release()
log_file.close()
cv2.destroyAllWindows()

### Step 7 — Check the log

In [12]:
import pandas as pd
pd.read_csv("output/waste_log.csv", names=["time", "class", "confidence", "bin"]).tail(10)

,time,class,confidence,bin
17,13:34:20,paper,0.99,Recyclable
18,13:34:25,paper,0.99,Recyclable
19,13:34:31,paper,0.94,Recyclable
20,13:34:36,paper,0.99,Recyclable
21,13:34:42,paper,0.96,Recyclable
22,13:34:47,paper,0.96,Recyclable
23,13:34:53,paper,1.00,Recyclable
24,13:34:59,paper,1.00,Recyclable
25,13:35:05,paper,0.79,Recyclable
26,13:35:12,plastic,0.97,Recyclable


### Summary
- Teachable Machine trains a small Dense head on a frozen, pre-trained MobileNet → Transfer Learning (Feature Extraction), same idea as Project A Task A6.2.
- Preprocessing (224×224, [-1, 1]) must match training or accuracy drops.
- Limitation: small, speed-optimised model; fine-tuning a bigger network would be more accurate.